# 🎬 AI Motion Transfer on Google Colab (A100 80GB)

> **Lưu ý về dung lượng:** Notebook này được cấu hình chạy trực tiếp trên đĩa SSD cục bộ của Colab (`/content/` có sẵn **150GB - 200GB miễn phí**), **KHÔNG CẦN DÙNG GOOGLE DRIVE** nên bạn không bao giờ lo bị giới hạn 15GB của Drive!

- **Mô hình cốt lõi:** Wan2.1-14B-I2V (Diffusion Transformer)
- **Kỹ thuật học:** LoRA (rank 128) + 3D Conv Pose Encoder + Multi-Objective Loss
- **Tối ưu tốc độ:** **Latent Consistency Distillation (LCD)** rút ngắn xuống **4-8 bước** (~15-20s / video)
- **Giao diện:** Gradio Web UI tạo link public truy cập trực tiếp (`share=True`)

--- 
## 1. Kiểm tra phần cứng GPU (A100 80GB)
Đảm bảo bạn đã chọn runtime: **Runtime -> Change runtime type -> A100 GPU**.

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device: {gpu_name}")
    print(f"Total VRAM: {vram_gb:.2f} GB")

--- 
## 2. Tải mã nguồn dự án về máy ảo Colab
Mã nguồn sẽ được clone về thư mục `/content/ai-motion-transfer` trên đĩa SSD của Colab.

In [ ]:
# Clone repository của bạn về Colab (hoặc upload thư mục zip)
# Thay đường dẫn repo GitHub của bạn vào dưới đây:
# !git clone https://github.com/YOUR_USERNAME/ai-motion-transfer.git /content/ai-motion-transfer
# %cd /content/ai-motion-transfer

# Tạo sẵn các thư mục cần thiết trên đĩa cục bộ Colab
!mkdir -p /content/data
!mkdir -p /content/checkpoints
!mkdir -p /content/checkpoints_distill
!mkdir -p /content/outputs

--- 
## 3. Cài đặt thư viện (Dependencies)

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers accelerate peft controlnet-aux opencv-python Pillow omegaconf einops decord imageio[ffmpeg] gradio
!pip install -q -e .
print('✅ Cài đặt môi trường hoàn tất!')

--- 
## 4. Chuẩn bị dữ liệu huấn luyện (Data Preparation)
Tải video nhảy và trích xuất pose bằng DWPose (lưu trực tiếp tại `/content/data`).

In [ ]:
# Tải và trích xuất pose cho video nhảy mẫu
!python prepare_data.py \
    --source aist++ \
    --output_dir /content/data \
    --max_videos 500 \
    --target_fps 15 \
    --target_resolution 832x480

# Xác thực tập dữ liệu
!python prepare_data.py --validate --output_dir /content/data

--- 
## 5. Giai đoạn 1: Huấn luyện Teacher Model (LoRA + PoseEncoder)
Huấn luyện mô hình chuẩn 25 bước trên A100 (lưu checkpoint vào `/content/checkpoints`).

In [ ]:
!accelerate launch train.py \
    --config config/default.yaml \
    --output_dir /content/checkpoints \
    --seed 42

--- 
## 6. Giai đoạn 2: Consistency Distillation (Nén xuống 8 bước ⚡)
Huấn luyện Student LoRA để rút ngắn thời gian sinh video xuống **~15-20 giây**.

In [ ]:
!accelerate launch train_distill.py \
    --config config/default.yaml \
    --teacher_checkpoint /content/checkpoints/final \
    --output_dir /content/checkpoints_distill \
    --num_steps 8

--- 
## 7. Khởi chạy Web UI (Gradio Public Link)
Giao diện trực quan cho phép upload ảnh và video để tạo chuyển động ngay lập tức.

In [ ]:
!python app.py

--- 
## 8. Tải Checkpoints về máy tính hoặc lưu lên Hugging Face (Không tốn Google Drive)
Sau khi train xong, bạn có thể nén model và tải thẳng về máy tính của mình mà không tốn dung lượng Google Drive!

In [ ]:
from google.colab import files

# Nén thư mục checkpoint đã distill thành file zip nhỏ gọn
!zip -r /content/distilled_model.zip /content/checkpoints_distill/final

# Tải trực tiếp file zip về máy tính cá nhân
files.download('/content/distilled_model.zip')